# simlytics — League Analytics

Run the standings SQL from `queries/` against the Postgres database and visualize it with seaborn / matplotlib.

Requires a `.env` at the project root with the `PG*` connection variables (the same ones `db/connection.py` reads).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make the project root importable and locate queries/, no matter what the
# kernel's working directory is.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
QUERIES = ROOT / 'queries'

from db.connection import connection

sns.set_theme(style='whitegrid', context='talk')


def run_query(sql: str, params=None) -> pd.DataFrame:
    """Execute a SQL string (optionally parameterized) and return a DataFrame."""
    with connection() as conn, conn.cursor() as cur:
        cur.execute(sql, params)
        cols = [d[0] for d in cur.description]
        rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)


def run_query_file(name: str, params=None) -> pd.DataFrame:
    """Run a .sql file from the queries/ directory (optionally parameterized)."""
    return run_query((QUERIES / name).read_text(), params)

## Driver standings
Gross championship points with per-driver summary stats.

In [ ]:
standings = run_query_file('driver_standings.sql')
standings.head(15)

In [ ]:
top = standings.head(15)
fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(data=top, x='points', y='driver_name', hue='driver_name',
            palette='viridis', legend=False, ax=ax)
ax.set(title='Driver Standings — 2026 Season 6', xlabel='Points', ylabel='')
for i, (pts, wins) in enumerate(zip(top['points'], top['wins'])):
    ax.text(pts + 3, i, f'{pts}  ({wins}W)', va='center', fontsize=12)
ax.margins(x=0.12)
plt.tight_layout()

## Points progression
Cumulative points after each round for the top 6 drivers — the title race over the season.

In [ ]:
prog = run_query_file('driver_points_progression.sql')
final_round = prog['round'].max()
leaders = (prog[prog['round'] == final_round]
           .nlargest(6, 'cumulative_points')['driver_name'])
sub = prog[prog['driver_name'].isin(leaders)]

fig, ax = plt.subplots(figsize=(12, 8))
sns.lineplot(data=sub, x='round', y='cumulative_points', hue='driver_name',
             marker='o', linewidth=2.2, ax=ax)
ax.set(title='Championship Points Progression (Top 6)',
       xlabel='Round', ylabel='Cumulative points')
ax.legend(title='', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

## What-if: position vs drop count
How each front-runner's championship position shifts as more results are dropped (P1 at the top).

In [ ]:
whatif = run_query_file('driver_standings_whatif.sql')
top10 = whatif.nsmallest(10, 'pos_d0')
pos_long = top10.melt(id_vars='driver_name',
                      value_vars=['pos_d0', 'pos_d1', 'pos_d2', 'pos_d3'],
                      var_name='drops', value_name='position')
pos_long['drops'] = pos_long['drops'].str.replace('pos_d', '').astype(int)

fig, ax = plt.subplots(figsize=(11, 8))
sns.lineplot(data=pos_long, x='drops', y='position', hue='driver_name',
             marker='o', linewidth=2.2, ax=ax)
ax.invert_yaxis()  # P1 at the top
ax.set(title='What-if: Championship Position vs Drop Count',
       xlabel='Number of drops', ylabel='Position', xticks=[0, 1, 2, 3])
ax.legend(title='', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

## Passing
Green-flag on-track passes only — pit-cycle and caution-period order changes are tagged in the `passes` table and excluded here.

In [ ]:
passing = run_query_file('passing_leaderboard.sql')
passing.head(10)

In [ ]:
top = passing.sort_values('passes_made', ascending=False).head(20)
fig, ax = plt.subplots(figsize=(11, 9))
sns.scatterplot(data=top, x='passes_made', y='passes_conceded',
                size='passes_defended', hue='net_passes', palette='coolwarm_r',
                sizes=(50, 500), ax=ax)
lim = max(top['passes_made'].max(), top['passes_conceded'].max()) * 1.05
ax.plot([0, lim], [0, lim], ls='--', color='grey', lw=1)  # break-even
ax.text(lim * 0.70, lim * 0.78, 'break even', color='grey', rotation=45, fontsize=11)
for _, r in top.iterrows():
    ax.annotate(r['driver_name'], (r['passes_made'], r['passes_conceded']),
                fontsize=9, xytext=(4, 4), textcoords='offset points')
ax.set(title='Passes Made vs Conceded (green flag)',
       xlabel='Passes made', ylabel='Passes conceded')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()

### Attack vs defense
Conversion rate (opportunities within 1% of a lap turned into a pass) vs defense rate (opportunities faced that were repelled). Dotted lines are the field medians; bubble size is the number of opportunities.

In [ ]:
conv = run_query_file('passing_conversion.sql')
cd = conv[conv['opportunities'] >= 50].copy()  # drivers with a real sample
cd[['conversion_pct', 'defense_pct']] = cd[['conversion_pct', 'defense_pct']].astype(float)

fig, ax = plt.subplots(figsize=(11, 8))
sns.scatterplot(data=cd, x='conversion_pct', y='defense_pct',
                size='opportunities', sizes=(60, 600), color='steelblue',
                alpha=0.7, ax=ax)
ax.axvline(cd['conversion_pct'].median(), ls=':', color='grey')
ax.axhline(cd['defense_pct'].median(), ls=':', color='grey')
for _, r in cd.iterrows():
    ax.annotate(r['driver_name'], (r['conversion_pct'], r['defense_pct']),
                fontsize=9, xytext=(4, 4), textcoords='offset points')
ax.set(title='Attack vs Defense — opportunity outcomes (green)',
       xlabel='Conversion %  (opportunities turned into passes)',
       ylabel='Defense %  (opportunities faced that were repelled)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()

### Battle chart — one driver's race
Gap to the car directly ahead each lap (as a % of a lap). Green triangles mark laps where the driver completed a green pass; gold bands are caution laps; the dashed line is the 1% opportunity threshold. Change `race_ss` / `driver` to explore other races.

In [ ]:
race_ss = 83742118
driver = 'Seth Hatchel'

gaps = run_query('''
    SELECT g.lap_num, g.gap_pct, g.under_caution
    FROM lap_gaps g JOIN drivers d ON d.cust_id = g.cust_id
    WHERE g.subsession_id = %(ss)s AND d.driver_name = %(drv)s AND g.same_lap
    ORDER BY g.lap_num''', {'ss': race_ss, 'drv': driver})
made = run_query('''
    SELECT p.lap_num
    FROM passes p JOIN drivers d ON d.cust_id = p.passer_cust_id
    WHERE p.subsession_id = %(ss)s AND d.driver_name = %(drv)s
      AND NOT p.pit_cycle AND NOT p.under_caution
    ORDER BY p.lap_num''', {'ss': race_ss, 'drv': driver})

g = gaps.dropna(subset=['gap_pct']).copy()
g['gap_pct'] = g['gap_pct'].astype(float) * 100

fig, ax = plt.subplots(figsize=(13, 6))
sns.lineplot(data=g, x='lap_num', y='gap_pct', color='steelblue', lw=1.5, ax=ax)
for lap in gaps.loc[gaps['under_caution'], 'lap_num']:
    ax.axvspan(lap - 0.5, lap + 0.5, color='gold', alpha=0.10)
ax.axhline(1.0, ls='--', color='grey', lw=1)
ax.scatter(made['lap_num'], [0.05] * len(made), marker='^', color='green',
           s=90, zorder=5, label='pass made')
ax.set(title=f'Battle chart — {driver} (subsession {race_ss})',
       xlabel='Lap', ylabel='Gap to car ahead (% of a lap)', ylim=(0, 4))
ax.legend(loc='upper right')
plt.tight_layout()

### Made vs conceded, by flag state
Diverging bars: passes **made** to the right, **conceded** to the left (hatched), each stacked by green / pit-cycle / caution. Green segments are the real racecraft; pit and caution segments are field churn.

In [ ]:
import numpy as np

flag = run_query_file('passing_by_flag.sql')
sel = flag.sort_values('total_made', ascending=False).head(12).iloc[::-1]
cats = ['green', 'pit', 'caution']
colors = {'green': '#2ca02c', 'pit': '#1f77b4', 'caution': '#ff7f0e'}
y = list(range(len(sel)))

fig, ax = plt.subplots(figsize=(12, 9))
left = np.zeros(len(sel))
for cat in cats:
    v = sel[f'made_{cat}'].astype(float).to_numpy()
    ax.barh(y, v, left=left, color=colors[cat], edgecolor='white', linewidth=0.5)
    left += v
left = np.zeros(len(sel))
for cat in cats:
    v = sel[f'conceded_{cat}'].astype(float).to_numpy()
    ax.barh(y, -v, left=left, color=colors[cat], edgecolor='white',
            linewidth=0.5, alpha=0.55, hatch='//')
    left -= v
ax.set_yticks(y)
ax.set_yticklabels(sel['driver_name'])
ax.axvline(0, color='black', lw=0.8)
ax.set(title='Passes Made (right) vs Conceded (left), by flag state',
       xlabel='conceded  (left)          passes          (right)  made')
from matplotlib.patches import Patch
handles = [Patch(facecolor=colors[c], label=c) for c in cats]
handles += [Patch(facecolor='grey', label='made'),
            Patch(facecolor='grey', alpha=0.55, hatch='//', label='conceded')]
ax.legend(handles=handles, loc='lower right', fontsize=9)
plt.tight_layout()

### Passing by track
Total passes per track (one race each), stacked by flag state, with the opportunity conversion rate annotated. Tracks that produce a lot of *green* passing race very differently from those dominated by pit/caution churn.

In [ ]:
import numpy as np

track = run_query_file('passing_by_track.sql')
tsel = track.sort_values('green_passes').copy()
colors = {'green': '#2ca02c', 'pit': '#1f77b4', 'caution': '#ff7f0e'}
labels = tsel['track_name'].str.strip()
y = list(range(len(tsel)))

fig, ax = plt.subplots(figsize=(12, 10))
left = np.zeros(len(tsel))
for col, c in [('green_passes', colors['green']),
               ('pit_passes', colors['pit']),
               ('caution_passes', colors['caution'])]:
    v = tsel[col].astype(float).to_numpy()
    ax.barh(y, v, left=left, color=c, edgecolor='white', linewidth=0.5,
            label=col.replace('_passes', ''))
    left += v
for i, (tot, conv) in enumerate(zip(left, tsel['conversion_pct'])):
    if conv is not None:
        ax.text(tot + left.max() * 0.01, i, f'{conv}% conv', va='center',
                fontsize=8, color='dimgrey')
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9)
ax.set(title='Passes by Track, split by flag state', xlabel='Passes (one race per track)')
ax.legend(title='flag', loc='lower right')
ax.margins(x=0.13)
plt.tight_layout()

## Restart performance
Net green passes per restart (made − conceded) during the first 2 laps after each restart — who capitalizes and who goes backwards. Limited to drivers with ≥ 20 restarts for a fair sample; the number in parentheses is that driver's restart count. Tune `min_restarts` in `passing_restarts.sql`.

In [ ]:
restarts = run_query_file('passing_restarts.sql')
reg = restarts[restarts['restarts'] >= 20].copy()
reg['net_per_restart'] = reg['net_per_restart'].astype(float)
reg = reg.sort_values('net_per_restart')

bar_colors = ['#2ca02c' if v >= 0 else '#d62728' for v in reg['net_per_restart']]
y = list(range(len(reg)))
fig, ax = plt.subplots(figsize=(12, 11))
ax.barh(y, reg['net_per_restart'], color=bar_colors, edgecolor='white')
ax.set_yticks(y)
ax.set_yticklabels(reg['driver_name'], fontsize=9)
ax.axvline(0, color='black', lw=0.8)
for i, (v, n) in enumerate(zip(reg['net_per_restart'], reg['restarts'])):
    ax.text(v + (0.03 if v >= 0 else -0.03), i, f'{v:+.2f} ({n})',
            va='center', ha='left' if v >= 0 else 'right', fontsize=8)
ax.set(title='Restart Performance — net positions gained per restart (≥ 20 restarts)',
       xlabel='Net green passes per restart  (made - conceded)')
ax.margins(x=0.16)
plt.tight_layout()

## Overall passing score
A weighted blend of four standardized (z-scored) passing skills — racecraft (net passes/race), attack (conversion %), defense (hold %), and restart (net/restart). Bars show each component's weighted **contribution** to the total; the bold number is the overall score. Adjust `weights` to re-rank (the z-scores come straight from `passing_score.sql`, so no re-query needed).

In [ ]:
import numpy as np

score = run_query_file('passing_score.sql')

# Weights must match the components; tweak to taste.
weights = {'net': 0.35, 'conv': 0.20, 'def': 0.20, 'restart': 0.25}
labels = {'net': 'racecraft', 'conv': 'attack', 'def': 'defense', 'restart': 'restart'}
comp_colors = {'net': '#4c72b0', 'conv': '#55a868', 'def': '#dd8452', 'restart': '#8172b3'}

contrib = pd.DataFrame({
    comp: score[f'z_{comp}'].astype(float) * w for comp, w in weights.items()
})
score['score'] = contrib.sum(axis=1)

top = score.sort_values('score', ascending=False).head(15).iloc[::-1]
ct = contrib.loc[top.index]
y = list(range(len(top)))

fig, ax = plt.subplots(figsize=(12, 10))
left_pos = np.zeros(len(top))
left_neg = np.zeros(len(top))
for comp in ['net', 'conv', 'def', 'restart']:
    v = ct[comp].to_numpy()
    pos = np.where(v > 0, v, 0.0)
    neg = np.where(v < 0, v, 0.0)
    ax.barh(y, pos, left=left_pos, color=comp_colors[comp], edgecolor='white',
            linewidth=0.5, label=labels[comp])
    ax.barh(y, neg, left=left_neg, color=comp_colors[comp], edgecolor='white',
            linewidth=0.5)
    left_pos += pos
    left_neg += neg
for i, v in enumerate(top['score']):
    ax.text(v + (0.03 if v >= 0 else -0.03), i, f'{v:+.2f}', va='center',
            ha='left' if v >= 0 else 'right', fontsize=9, fontweight='bold')
ax.set_yticks(y)
ax.set_yticklabels(top['driver_name'])
ax.axvline(0, color='black', lw=0.8)
ax.set(title='Overall Passing Score — weighted component contributions (top 15)',
       xlabel='score contribution (weighted z-score)')
ax.legend(title='component', loc='lower right', fontsize=9)
plt.tight_layout()

## Does passing translate to results?
Correlate each driver's overall passing score against their average finishing position (1 = win, so a **negative** correlation means better passers finish higher). The regression line and Pearson r / Spearman rho quantify the link; the correlation with championship points is printed too.

In [ ]:
score = run_query_file('passing_score.sql')[['season_name', 'driver_name', 'passing_score']]
stand = run_query_file('driver_standings.sql')[['season_name', 'driver_name',
                                                 'avg_finish', 'points', 'pos']]
m = score.merge(stand, on=['season_name', 'driver_name'], how='inner')
for col in ['passing_score', 'avg_finish', 'points']:
    m[col] = m[col].astype(float)

r_fin = m['passing_score'].corr(m['avg_finish'])
rho_fin = m['passing_score'].corr(m['avg_finish'], method='spearman')
r_pts = m['passing_score'].corr(m['points'])
print(f'passing score vs avg finish:  Pearson r = {r_fin:.2f}, Spearman rho = {rho_fin:.2f}')
print(f'passing score vs points:      Pearson r = {r_pts:.2f}')

fig, ax = plt.subplots(figsize=(11, 8))
sns.regplot(data=m, x='passing_score', y='avg_finish', ax=ax,
            scatter_kws={'s': 70, 'alpha': 0.7, 'color': 'steelblue'},
            line_kws={'color': 'crimson'})
ax.invert_yaxis()  # better finish (lower number) at the top
for _, r in m.iterrows():
    ax.annotate(r['driver_name'], (r['passing_score'], r['avg_finish']),
                fontsize=8, xytext=(4, 3), textcoords='offset points')
ax.set(title=f'Passing Score vs Average Finish  (r = {r_fin:.2f}, rho = {rho_fin:.2f})',
       xlabel='Overall passing score', ylabel='Average finishing position (1 = win)')
ax.text(0.02, 0.02, f'vs championship points:  r = {r_pts:.2f}',
        transform=ax.transAxes, fontsize=10, color='dimgrey')
plt.tight_layout()

## Green-flag pit-cycle times
Median time lost in a green-flag pit cycle (in-lap + out-lap, minus what those laps would have taken at green pace), for stops made in a real pit window (a significant share of the field pitting within a few laps). Lower = faster through the pits. Black dots mark each driver's single best stop; the label shows their green-stop count. The x-axis starts near 30s to make the spread legible. From `pit_cycle_ranking.sql`.

In [ ]:
pit = run_query_file('pit_cycle_ranking.sql')
for col in ['median_time_lost_s', 'best_time_lost_s', 'median_cycle_s']:
    pit[col] = pit[col].astype(float)
pit = pit.sort_values('median_time_lost_s', ascending=False)  # fastest ends up on top
y = list(range(len(pit)))

norm = plt.Normalize(pit['median_time_lost_s'].min(), pit['median_time_lost_s'].max())
bar_colors = plt.cm.RdYlGn_r(norm(pit['median_time_lost_s']))

fig, ax = plt.subplots(figsize=(11, 11))
ax.barh(y, pit['median_time_lost_s'], color=bar_colors, edgecolor='white')
ax.scatter(pit['best_time_lost_s'], y, color='black', s=25, zorder=5, label='best stop')
ax.set_yticks(y)
ax.set_yticklabels([f'{n}  ({s})' for n, s in zip(pit['driver_name'], pit['green_stops'])],
                   fontsize=8)
ax.set_xlim(left=30)
for i, v in enumerate(pit['median_time_lost_s']):
    ax.text(v + 0.1, i, f'{v:.1f}s', va='center', fontsize=8)
ax.set(title='Green-Flag Pit-Cycle Time Lost (median; lower = faster)',
       xlabel='Time lost vs normal green laps (s)  [axis starts at 30s]')
ax.legend(loc='lower right')
plt.tight_layout()

## Pit cycles for a single race
The season view above aggregates every race. To drill into one race, pick a `subsession_id` from the list below and pass it to the parameterized `pit_cycle_by_race.sql`.

In [ ]:
# Races with green pit-window data to choose from.
races_avail = run_query('''
    SELECT ra.subsession_id, t.track_name, t.track_config_name,
           count(*) AS green_window_stops
    FROM pit_cycles pc
    JOIN races ra ON ra.subsession_id = pc.subsession_id
    JOIN tracks t ON t.track_id = ra.track_id
    WHERE pc.in_green_window
    GROUP BY ra.subsession_id, t.track_name, t.track_config_name
    ORDER BY ra.subsession_id''')
races_avail

In [ ]:
race_ss = 85968723  # <-- set to any subsession_id from races_avail

race_pits = run_query_file('pit_cycle_by_race.sql', {'subsession_id': race_ss})
race_pits['time_lost_s'] = race_pits['time_lost_s'].astype(float)
track_name = races_avail.loc[races_avail['subsession_id'] == race_ss, 'track_name']
track_name = track_name.iloc[0].strip() if len(track_name) else str(race_ss)

# best (fastest) green stop per driver in this race
best = (race_pits.groupby('driver_name')
        .agg(time_lost_s=('time_lost_s', 'min'), stops=('stop_num', 'count'))
        .reset_index().sort_values('time_lost_s', ascending=False))
y = list(range(len(best)))

norm = plt.Normalize(best['time_lost_s'].min(), best['time_lost_s'].max())
bar_colors = plt.cm.RdYlGn_r(norm(best['time_lost_s']))
fig, ax = plt.subplots(figsize=(11, max(4, 0.34 * len(best))))
ax.barh(y, best['time_lost_s'], color=bar_colors, edgecolor='white')
ax.set_yticks(y)
ax.set_yticklabels([f'{n}  ({s})' for n, s in zip(best['driver_name'], best['stops'])],
                   fontsize=8)
for i, v in enumerate(best['time_lost_s']):
    ax.text(v + 0.2, i, f'{v:.1f}s', va='center', fontsize=8)
ax.set_xlim(left=max(0, best['time_lost_s'].min() - 3))
ax.set(title=f'Green Pit-Cycle Time Lost — {track_name} (subsession {race_ss})',
       xlabel='Best green pit-cycle time lost in this race (s)')
plt.tight_layout()

## Driver feature vectors & lap-time distributions
Green racing pace is normalized to each race's median lap so tracks with different lap lengths are comparable: `pace_pct` = percent off the field's median lap that race (negative = faster than the field). Per driver we fit a skew-normal to that distribution and assemble a feature vector combining pace, consistency, passing skill (z-scores), pit speed, and results.

In [ ]:
from scipy import stats as sstats
import numpy as np

from stats.driver_features import green_lap_pace, build_features

# Green-lap pace (per lap, track-normalized) and the assembled feature vector
# both come from stats/driver_features.py, so other analyses can reuse them.
with connection() as conn:
    gl = green_lap_pace(conn)
    features = build_features(conn, gl=gl)

feat_cols = ['driver_name', 'green_laps', 'pace_pct_median', 'pace_pct_std',
             'pace_pct_skew', 'avg_finish', 'incidents_per_race', 'passing_score',
             'pit_median_lost_s']
features[features['green_laps'] >= 300][feat_cols].sort_values('pace_pct_median').round(2)

### Fitted pace distributions
Skew-normal fits for the fastest, most consistent, and a midfield driver (histogram = observed, curve = fit). Further left = faster; narrower = more consistent.

In [ ]:
regs = features[features['green_laps'] >= 300].copy()
fastest = regs.loc[regs['pace_pct_median'].idxmin(), 'driver_name']
consistent = regs.loc[regs['pace_pct_std'].idxmin(), 'driver_name']
mid = regs.sort_values('pace_pct_median').iloc[len(regs) // 2]['driver_name']
sel = list(dict.fromkeys([fastest, consistent, mid]))

sub = gl[gl['driver_name'].isin(sel)]['pace_pct']
x = np.linspace(sub.min(), sub.max(), 400)
palette = sns.color_palette('deep', len(sel))
fig, ax = plt.subplots(figsize=(11, 7))
for drv, color in zip(sel, palette):
    v = gl[gl['driver_name'] == drv]['pace_pct']
    sns.histplot(v, stat='density', bins=40, alpha=0.20, color=color, ax=ax)
    a, loc, scale = features.loc[features['driver_name'] == drv,
                                 ['skewnorm_a', 'skewnorm_loc', 'skewnorm_scale']].iloc[0]
    ax.plot(x, sstats.skewnorm.pdf(x, a, loc, scale), color=color, lw=2.4,
            label=f'{drv}  (med {v.median():+.2f}%, sd {v.std():.2f})')
ax.set(title='Green-lap pace distributions (fitted skew-normal)',
       xlabel='Pace vs race median (%) - left = faster', ylabel='density')
ax.legend()
plt.tight_layout()

In [ ]:
# Pairwise distribution distance (two-sample KS) among the most-raced drivers.
topN = features.nlargest(14, 'green_laps')['driver_name'].tolist()
data = {d: gl[gl['driver_name'] == d]['pace_pct'].to_numpy() for d in topN}
n = len(topN)
ks = np.zeros((n, n))
for i, a in enumerate(topN):
    for j, b in enumerate(topN):
        ks[i, j] = sstats.ks_2samp(data[a], data[b]).statistic
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(ks, xticklabels=topN, yticklabels=topN, cmap='mako_r',
            cbar_kws={'label': 'KS statistic (0 = identical, 1 = disjoint)'}, ax=ax)
ax.set(title='Pairwise pace-distribution distance (two-sample KS)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()

### Who is significantly faster? (Mann-Whitney)
For every pair, a one-sided Mann-Whitney U tests whether one driver's green laps are stochastically faster. With thousands of laps the test is easily 'significant', so we also require a **>=0.1% median pace edge** to count it as practically faster. Bars = how many rivals each driver clears on both.

In [ ]:
regs_list = features[features['green_laps'] >= 300]['driver_name'].tolist()
d = {x: gl[gl['driver_name'] == x]['pace_pct'].to_numpy() for x in regs_list}
med = {x: np.median(d[x]) for x in regs_list}
faster = {}
for a in regs_list:
    cnt = 0
    for b in regs_list:
        if a == b:
            continue
        p = sstats.mannwhitneyu(d[a], d[b], alternative='less').pvalue
        if p < 0.05 and med[a] < med[b] - 0.1:
            cnt += 1
    faster[a] = cnt
mw = pd.Series(faster).sort_values()
n_rivals = len(regs_list) - 1

fig, ax = plt.subplots(figsize=(11, max(4, 0.34 * len(mw))))
ax.barh(range(len(mw)), mw.values,
        color=plt.cm.viridis(np.linspace(0.15, 0.9, len(mw))), edgecolor='white')
ax.set_yticks(range(len(mw)))
ax.set_yticklabels(mw.index, fontsize=8)
for i, v in enumerate(mw.values):
    ax.text(v + 0.1, i, str(v), va='center', fontsize=8)
ax.set(title=f'Pace dominance - rivals each driver is significantly faster than (of {n_rivals})',
       xlabel='rivals cleared (Mann-Whitney p<0.05 and >=0.1% median edge)')
plt.tight_layout()